# Week 5 — Stress (PSM-25) on MCD-rPPG: rPPG + audio, transformers, literature comparison

Builds on weeks 3–4. It reuses the saved rPPG features, the 30 Hz signal cache and the within-person features, so **no video re-extraction is needed**.

**Supervisor suggestions → where they are in this notebook:**

| Suggestion | Section |
|---|---|
| rPPG + audio, tried first on MCD | 1 (does MCD have audio?), 2 (audio features), 3 (recording-context control), 4 (rPPG + audio ablation) |
| Transformer | 5 (pulse transformer, audio transformer, rPPG + audio fusion transformer) |
| Compare with literature | 6 |
| Proceed with another dataset | 7 (readiness check for StressID / UBFC-Phys) |

**Important context:** in MCD-rPPG, participants sat and *waited motionlessly for three minutes* (paper, Sec. 2). Nobody speaks, so any audio track holds room noise, breathing and the operator, not speech. Section 1 checks whether the videos have audio at all. If they don't, sections 2–4 are skipped automatically and the notebook still runs the transformer and literature parts.

**Evaluation (unchanged from weeks 3–4):**
- Unit of analysis: the patient.
- Fixed split: `train_test_split(..., test_size=0.2, random_state=43)`, constant MAE = 1.117.
- Repeated 5-fold CV with patient-bootstrap 95% CIs.
- A feature set "helps" only if its gain CI vs demographics is entirely above 0.

**Runtime:**
- Section 2 (audio): about 30–60 min, resumable.
- Section 4: about 15–25 min.
- Section 5: about 10–20 min on a GPU (Runtime → Change runtime type → T4).
- Everything else takes minutes.

## 0. Setup and reload of weeks 3–4 results

In [1]:
# Keeps Colab's own versions; never upgrade these with -U.
!pip -q install "pandas==2.2.3" "huggingface_hub<2"
import pandas, huggingface_hub, subprocess, shutil
print('pandas', pandas.__version__, '| huggingface_hub', huggingface_hub.__version__,
      '| ffmpeg:', shutil.which('ffmpeg') is not None)

pandas 2.2.3 | huggingface_hub 1.33.0 | ffmpeg: True


In [2]:
import os, re, json, math, time, warnings, subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---- paths: same Drive layout as weeks 1-4 ----
ROOT      = Path('/content/drive/MyDrive/MCD_rPPG') if IN_COLAB else Path('./MCD_rPPG')
DATA_ROOT = ROOT / 'raw'
DB_PATH   = ROOT / 'metadata' / 'db.csv'
W3_FEAT   = ROOT / 'week3_stress_rppg600' / 'features'      # week 3-4 outputs (read only)
WORK      = ROOT / 'week5_stress_multimodal'
FEAT_DIR  = WORK / 'features'
RES_DIR   = WORK / 'results'
AUD_DIR   = WORK / 'audio_cache'                            # one small .npz per session
TMP       = Path('/content/mcd_tmp') if IN_COLAB else Path('./mcd_tmp')
for p in (WORK, FEAT_DIR, RES_DIR, AUD_DIR, TMP):
    p.mkdir(parents=True, exist_ok=True)

REPO_ID    = 'wengziheng/mcd_rppg'
CAMERA     = 'FullHDwebcam'
FS_RPPG    = 30.0
HR_BAND    = (0.7, 3.5)
SPLIT_SEED = 43
N_REPEATS  = 10
N_FOLDS    = 5
N_BOOT     = 2000
N_WORKERS  = max(1, os.cpu_count() or 1)
for f in ('session_features.csv', 'signals_30hz.npz', 'within_person_features.csv'):
    print(f'{f:30s}', 'found' if (W3_FEAT / f).exists() else 'MISSING (run the week 3-4 notebook first)')

Mounted at /content/drive
session_features.csv           found
signals_30hz.npz               found
within_person_features.csv     found


In [3]:
from huggingface_hub import hf_hub_download, HfApi
try:
    from huggingface_hub import get_token
    _saved = get_token()
except Exception:
    _saved = None
if not _saved:
    from huggingface_hub import login; login()        # only needed if videos must be downloaded
    _saved = get_token()

def _token_ok(tok):
    try:
        return HfApi().whoami(token=tok).get('name')
    except Exception:
        return None
TOKEN_ARG = _saved if (_saved and _token_ok(_saved)) else False
print('HF token:', 'ok' if TOKEN_ARG else 'none (anonymous, rate-limited)')

def _status(e):
    code = getattr(getattr(e, 'response', None), 'status_code', None)
    if code is None:
        m = re.search(r'\b(401|403|404|429)\b', str(e).split('Request ID')[0])
        code = int(m.group(1)) if m else None
    return code

def fetch(relpath, keep=True, tries=6):
    # Local path for a repo file: reuse Drive copy, else download (to Drive if keep, else to local TMP).
    p = DATA_ROOT / relpath
    if p.exists() and p.stat().st_size > 0:
        return p
    q = TMP / relpath
    if q.exists() and q.stat().st_size > 0:
        return q
    dest = DATA_ROOT if keep else TMP
    for attempt in range(tries):
        try:
            return Path(hf_hub_download(REPO_ID, relpath, repo_type='dataset',
                                        local_dir=str(dest), token=TOKEN_ARG))
        except Exception as e:
            code = _status(e)
            if code in (401, 403) or type(e).__name__ in ('GatedRepoError', 'RepositoryNotFoundError'):
                raise RuntimeError(f'Access denied for {REPO_ID}/{relpath} (HTTP {code}).') from e
            if attempt == tries - 1:
                raise
            time.sleep((30 if code == 429 else 5) * (attempt + 1))

HF token: ok


In [4]:
# ---- labels, week-3 session features, week-4 within-person features, 30 Hz signal cache ----
from scipy.signal import butter, filtfilt, welch
from numpy.lib.stride_tricks import sliding_window_view

def bandpass(x, fs, lo, hi, order=3):
    b, a = butter(order, [lo / (fs / 2), hi / (fs / 2)], btype='band')
    return filtfilt(b, a, x)

db = pd.read_csv(DB_PATH if DB_PATH.exists() else fetch('db.csv'))
front = db[db.camera == CAMERA].copy().reset_index(drop=True)
sess = pd.read_csv(W3_FEAT / 'session_features.csv')
sess['rppg_good'] = (sess.rppg_snr > 0) & (sess.rppg_hr_consistency > 0.6)

VITALS = ['upper_ap', 'lower_ap', 'saturation', 'respiratory', 'rigidity', 'pulse']
def build_patient_table(sess, front):
    base = front.sort_values('step').groupby('patient_id')
    P = pd.DataFrame({'stress': base.stress.mean(), 'age': base.age.first(),
                      'sex_m': base.sex.first().map({'M': 1, 'F': 0}), 'bmi': base.bmi.first()})
    rest = front[front.step == 'before'].set_index('patient_id')
    for v in VITALS:
        P[v] = rest[v]
    feat_cols = [c for c in sess.columns if c.split('_')[0] in ('rppg', 'ppg', 'ecg')
                 and not c.endswith('_error') and c != 'rppg_good' and pd.api.types.is_numeric_dtype(sess[c])]
    b = sess[sess.step == 'before'].set_index('patient_id')[feat_cols]
    a = sess[sess.step == 'after'].set_index('patient_id')[feat_cols]
    P = P.join(b.add_suffix('_rest')).join(a.add_suffix('_post'))
    for c in feat_cols:
        P[c + '_avg'] = P[[c + '_rest', c + '_post']].mean(axis=1)
    for src in ('rppg', 'ppg', 'ecg'):
        if f'{src}_hr_rest' in P:
            P[f'{src}_hr_resp'] = P[f'{src}_hr_post'] - P[f'{src}_hr_rest']
    return P.dropna(subset=['stress'])

P = build_patient_table(sess, front)
W = pd.read_csv(W3_FEAT / 'within_person_features.csv', index_col=0)
P = P.join(W)

DEMO = ['age', 'sex_m', 'bmi']
def core(src, kind='rest'):
    cols = [f'{src}_hr_rest', f'{src}_ln_rmssd_{kind}', f'{src}_ln_sdnn_{kind}', f'{src}_ln_hf_{kind}', f'{src}_hr_resp']
    return [c for c in cols if c in P.columns]
RPPG_WITHIN = [c for c in W.columns if c.startswith('rppg_w_')]
RPPG_ALL = core('rppg') + RPPG_WITHIN

with np.load(W3_FEAT / 'signals_30hz.npz') as zc:
    SIG = {k: zc[k] for k in zc.files}
print('patients:', len(P), '| rPPG features:', len(RPPG_ALL), '| cached pulse signals:', len(SIG))

patients: 600 | rPPG features: 13 | cached pulse signals: 2398


In [5]:
# ---- evaluation helpers (identical to weeks 3-4) ----
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegressionCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata, norm, spearmanr

ALPHAS = np.logspace(-2, 4, 25)
def ridge(alphas=ALPHAS):
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), RidgeCV(alphas=alphas))

class DemoResidual(BaseEstimator, RegressorMixin):
    # Demographics Ridge first; a heavily shrunk Ridge on the other features predicts the residual.
    def __init__(self, demo=('age', 'sex_m', 'bmi')):
        self.demo = demo
    def fit(self, X, y):
        X = pd.DataFrame(X)
        self.d_ = [c for c in X.columns if c in self.demo]
        self.r_ = [c for c in X.columns if c not in self.demo]
        self.m1_ = ridge().fit(X[self.d_], y)
        res = y - self.m1_.predict(X[self.d_])
        self.m2_ = ridge(np.logspace(1, 5, 20)).fit(X[self.r_], res) if self.r_ else None
        return self
    def predict(self, X):
        X = pd.DataFrame(X)
        p = self.m1_.predict(X[self.d_])
        return p + self.m2_.predict(X[self.r_]) if self.m2_ is not None else p

MODELS = {
    'Ridge':    lambda: ridge(),
    'RF':       lambda: make_pipeline(SimpleImputer(strategy='median'),
                                      RandomForestRegressor(200, min_samples_leaf=20, max_features=0.5,
                                                            n_jobs=-1, random_state=0)),
    'HGB':      lambda: HistGradientBoostingRegressor(loss='absolute_error', max_depth=3, learning_rate=0.03,
                                                      max_iter=300, min_samples_leaf=30, l2_regularization=1.0,
                                                      random_state=0),
    'Residual': lambda: DemoResidual(),
}

def oof_predictions(X, y, make, n_rep=N_REPEATS, n_folds=N_FOLDS, seed0=100):
    preds = np.zeros((n_rep, len(y)))
    for r in range(n_rep):
        for tr, te in KFold(n_folds, shuffle=True, random_state=seed0 + r).split(y):
            if make is None:
                preds[r, te] = np.median(y[tr])
            else:
                preds[r, te] = make().fit(X.iloc[tr], y[tr]).predict(X.iloc[te])
    return preds

def boot_skill(e, e0, e_ref=None, n_boot=N_BOOT, seed=1):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(e), (n_boot, len(e)))
    s = 1 - e[idx].sum(1) / e0[idx].sum(1)
    out = dict(skill=1 - e.sum() / e0.sum(), skill_lo=np.percentile(s, 2.5), skill_hi=np.percentile(s, 97.5))
    if e_ref is not None:
        g = s - (1 - e_ref[idx].sum(1) / e0[idx].sum(1))
        out.update(gain=(1 - e.sum() / e0.sum()) - (1 - e_ref.sum() / e0.sum()),
                   gain_lo=np.percentile(g, 2.5), gain_hi=np.percentile(g, 97.5))
    return out

def run_ladder(df, sets, models=('Ridge', 'RF', 'Residual')):
    # Fixed split (rs=43) test MAE + repeated-CV skill vs constant and gain vs demographics (Ridge).
    y, ids = df.stress.values, df.index.values
    tr_ids, te_ids = train_test_split(ids, test_size=0.2, random_state=SPLIT_SEED)
    tr, te = np.isin(ids, tr_ids), np.isin(ids, te_ids)
    e0 = np.abs(oof_predictions(None, y, None) - y).mean(0)
    eref = np.abs(oof_predictions(df[DEMO], y, MODELS['Ridge']) - y).mean(0)
    rows = [dict(set='constant (train median)', model='-', n_feat=0,
                 test_mae=np.mean(np.abs(y[te] - np.median(y[tr]))), cv_mae=e0.mean())]
    for sname, cols in tqdm(sets.items()):
        if not cols:
            continue
        has_demo = any(c in DEMO for c in cols)
        for mname in models:
            if mname == 'Residual' and (not has_demo or all(c in DEMO for c in cols)):
                continue
            make = MODELS[mname]
            test_mae = np.mean(np.abs(y[te] - make().fit(df.loc[tr, cols], y[tr]).predict(df.loc[te, cols])))
            e = np.abs(oof_predictions(df[cols], y, make) - y).mean(0)
            rows.append(dict(set=sname, model=mname, n_feat=len(cols), test_mae=test_mae, cv_mae=e.mean(),
                             **{k: 100 * v for k, v in boot_skill(e, e0, eref).items()}))
    return pd.DataFrame(rows)

def logit():
    return make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                         LogisticRegressionCV(Cs=np.logspace(-3, 2, 12), cv=5, scoring='roc_auc', max_iter=2000))

def oof_proba(X, yb, n_rep=N_REPEATS, seed0=200):
    pr = np.zeros((n_rep, len(yb)))
    for r in range(n_rep):
        for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed0 + r).split(X, yb):
            pr[r, te] = logit().fit(X.iloc[tr], yb[tr]).predict_proba(X.iloc[te])[:, 1]
    return pr.mean(0)

def boot_auc(yb, p, p_ref, n_boot=N_BOOT, seed=2):
    rng = np.random.default_rng(seed)
    a, d = [], []
    for _ in range(n_boot):
        i = rng.integers(0, len(yb), len(yb))
        if yb[i].min() == yb[i].max():
            continue
        a.append(roc_auc_score(yb[i], p[i])); d.append(a[-1] - roc_auc_score(yb[i], p_ref[i]))
    return dict(auc=roc_auc_score(yb, p), auc_lo=np.percentile(a, 2.5), auc_hi=np.percentile(a, 97.5),
                d_auc=roc_auc_score(yb, p) - roc_auc_score(yb, p_ref),
                d_lo=np.percentile(d, 2.5), d_hi=np.percentile(d, 97.5))

def partial_spearman(df, x, y='stress', cov=DEMO):
    d = df[[x, y] + cov].dropna()
    R = d.apply(rankdata)
    Z = R[cov].values
    rx = R[x] - LinearRegression().fit(Z, R[x]).predict(Z)
    ry = R[y] - LinearRegression().fit(Z, R[y]).predict(Z)
    rho = np.corrcoef(rx, ry)[0, 1]
    se = 1 / np.sqrt(max(len(d) - len(cov) - 3, 1))
    z = np.arctanh(rho)
    return dict(feature=x, n=len(d), rho=rho, lo=np.tanh(z - 1.96 * se), hi=np.tanh(z + 1.96 * se),
                p=2 * (1 - norm.cdf(abs(z) / se)))

def bh(p):
    p = np.asarray(p); o = np.argsort(p); q = p[o] * len(p) / np.arange(1, len(p) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q); out[o] = np.minimum(q, 1); return out
print('helpers ready')

helpers ready


## 1. Does MCD-rPPG have audio?

Probes the audio stream (ffprobe) of a few front-camera videos already on Drive, plus one video from each of the other two cameras, downloaded temporarily and then deleted. The camera with audio is used next, with the front camera preferred.

If **no camera has audio**, `HAS_AUDIO = False`. Sections 2–4 then report that and skip the audio work, and the rPPG + audio idea has to move to a dataset with audio (StressID, section 7).

In [6]:
def probe_audio(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-select_streams', 'a',
                        '-show_entries', 'stream=codec_name,sample_rate,channels:format=duration',
                        '-of', 'json', str(path)], capture_output=True, text=True)
    info = json.loads(r.stdout or '{}')
    streams = info.get('streams', [])
    return dict(has_audio=len(streams) > 0,
                codec=streams[0].get('codec_name') if streams else None,
                sample_rate=streams[0].get('sample_rate') if streams else None,
                duration=info.get('format', {}).get('duration'))

probe_rows = []
for cam in ('FullHDwebcam', 'USBVideo', 'IriunWebcam'):
    rows = db[db.camera == cam]
    if rows.empty:
        continue
    local = [r for r in rows.itertuples() if (DATA_ROOT / r.video).exists()][:3]
    picks = [(r, False) for r in local] or [(rows.iloc[0], True)]
    for r, temporary in picks:
        try:
            path = fetch(r.video, keep=False) if temporary else DATA_ROOT / r.video
            probe_rows.append(dict(camera=cam, video=Path(r.video).name, **probe_audio(path)))
            if temporary and str(path).startswith(str(TMP)):
                Path(path).unlink(missing_ok=True)
        except Exception as e:
            probe_rows.append(dict(camera=cam, video=Path(r.video).name, has_audio=None, codec=repr(e)[:80]))
probe = pd.DataFrame(probe_rows)
print(probe.to_string(index=False))

cams_with_audio = [c for c in ('FullHDwebcam', 'USBVideo', 'IriunWebcam')
                   if probe[(probe.camera == c) & (probe.has_audio == True)].shape[0] > 0]
HAS_AUDIO = len(cams_with_audio) > 0
AUDIO_CAMERA = cams_with_audio[0] if HAS_AUDIO else None
print('\nHAS_AUDIO =', HAS_AUDIO, '| audio camera:', AUDIO_CAMERA)
if not HAS_AUDIO:
    print('MCD videos carry no audio track -> rPPG + audio cannot be tested on MCD. '
          'Sections 2-4 will skip the audio parts; use StressID (has audio) for this idea.')

      camera                        video  has_audio codec sample_rate duration
FullHDwebcam  1024_FullHDwebcam_after.avi      False  None        None     None
FullHDwebcam 1024_FullHDwebcam_before.avi      False  None        None     None
FullHDwebcam  1091_FullHDwebcam_after.avi      False  None        None     None
    USBVideo      1543_USBVideo_after.avi      False  None        None     None
    USBVideo     1543_USBVideo_before.avi      False  None        None     None
    USBVideo      1589_USBVideo_after.avi      False  None        None     None
 IriunWebcam   1543_IriunWebcam_after.avi      False  None        None     None
 IriunWebcam  1543_IriunWebcam_before.avi      False  None        None     None
 IriunWebcam   1589_IriunWebcam_after.avi      False  None        None     None

HAS_AUDIO = False | audio camera: None
MCD videos carry no audio track -> rPPG + audio cannot be tested on MCD. Sections 2-4 will skip the audio parts; use StressID (has audio) for this idea.


## 2. Audio features (only if audio exists)

Per session, using ffmpeg to decode to 16 kHz mono, then numpy/scipy (no extra packages):
- **Loudness:** frame RMS (dB) mean, SD and dynamic range; fraction of active frames.
- **Spectral shape:** centroid, bandwidth, roll-off, flatness, zero-crossing rate (mean and SD).
- **MFCC 0–12:** mean and SD.
- **Breathing:** the 200–2000 Hz energy envelope is band-passed to 0.1–0.7 Hz, giving a breathing rate (breaths/min) and peak strength. This is checked against the measured respiratory rate in `db.csv`, which shows whether the audio actually captures breathing.
- **Per-second log-mel (40 bands):** kept for the transformers in section 5.

Each session is saved to `audio_cache/<patient>_<step>.npz`, so the cell is resumable. Videos already on Drive are read directly; missing ones are downloaded temporarily and deleted.

In [7]:
from scipy.fft import dct
SR = 16000

def read_audio(path, sr=SR):
    r = subprocess.run(['ffmpeg', '-v', 'error', '-i', str(path), '-vn', '-ac', '1', '-ar', str(sr),
                        '-f', 'f32le', '-'], capture_output=True)
    if r.returncode != 0 or len(r.stdout) < 4 * sr * 10:
        return None
    return np.frombuffer(r.stdout, np.float32).astype(np.float64)

def mel_filterbank(n_mels=40, n_fft=512, sr=SR, fmin=50, fmax=8000):
    hz2mel = lambda f: 2595 * np.log10(1 + f / 700)
    mel2hz = lambda m: 700 * (10 ** (m / 2595) - 1)
    bins = np.floor((n_fft + 1) * mel2hz(np.linspace(hz2mel(fmin), hz2mel(fmax), n_mels + 2)) / sr).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        l, c, r = bins[m - 1], bins[m], bins[m + 1]
        fb[m - 1, l:c] = (np.arange(l, c) - l) / max(c - l, 1)
        fb[m - 1, c:r] = (r - np.arange(c, r)) / max(r - c, 1)
    return fb
MEL_FB = mel_filterbank()

def audio_features(x, sr=SR, n_fft=512, hop=160):
    eps = 1e-12
    x = x - x.mean()
    frames = sliding_window_view(x, n_fft)[::hop] * np.hanning(n_fft)       # 32 ms frames, 10 ms hop
    spec = np.abs(np.fft.rfft(frames, axis=1)) ** 2
    freqs = np.fft.rfftfreq(n_fft, 1 / sr)
    rms_db = 10 * np.log10(np.mean(frames ** 2, axis=1) + eps)
    mel = np.log(spec @ MEL_FB.T + eps)
    mfcc = dct(mel, type=2, axis=1, norm='ortho')[:, :13]
    p = spec / (spec.sum(1, keepdims=True) + eps)
    centroid = (p * freqs).sum(1)
    bandwidth = np.sqrt((p * (freqs - centroid[:, None]) ** 2).sum(1))
    rolloff = freqs[np.argmax(np.cumsum(p, 1) >= 0.85, axis=1)]
    flatness = np.exp(np.mean(np.log(spec + eps), 1)) / (np.mean(spec, 1) + eps)
    zcr = np.mean(np.abs(np.diff(np.sign(frames), axis=1)) > 0, axis=1)
    floor = np.percentile(rms_db, 10)
    F = dict(rms_mean=rms_db.mean(), rms_sd=rms_db.std(), rms_range=np.percentile(rms_db, 90) - floor,
             active_frac=np.mean(rms_db > floor + 10))
    for name, v in (('centroid', centroid), ('bandwidth', bandwidth), ('rolloff', rolloff),
                    ('flatness', flatness), ('zcr', zcr)):
        F[name + '_mean'], F[name + '_sd'] = v.mean(), v.std()
    for i in range(13):
        F[f'mfcc{i}_mean'], F[f'mfcc{i}_sd'] = mfcc[:, i].mean(), mfcc[:, i].std()
    # breathing: slow modulation of the 200-2000 Hz energy
    fs_env = sr / hop
    env = np.log(spec[:, (freqs >= 200) & (freqs <= 2000)].sum(1) + eps)
    env = bandpass(env - env.mean(), fs_env, 0.1, 0.7)
    f, Pw = welch(env, fs_env, nperseg=min(len(env), int(60 * fs_env)))
    m = (f >= 0.1) & (f <= 0.7)
    F['resp_rate'] = 60 * f[m][np.argmax(Pw[m])]
    F['resp_peak'] = Pw[m].max() / (np.median(Pw[m]) + eps)
    T = len(mel) // 100
    return F, mel[:T * 100].reshape(T, 100, -1).mean(1).astype(np.float16)    # per-second log-mel

def audio_one(r):
    out = AUD_DIR / f"{r['patient_id']}_{r['step']}.npz"
    if out.exists():
        return 'skip'
    path, temporary = DATA_ROOT / r['video'], False
    try:
        if not path.exists():
            path, temporary = fetch(r['video'], keep=False), True
        x = read_audio(path)
        if x is None:
            return 'no audio'
        F, mel = audio_features(x)
        np.savez_compressed(out, names=np.array(list(F)), vals=np.array(list(F.values()), float), mel=mel)
        return 'ok'
    except Exception as e:
        return 'error: ' + repr(e)[:120]
    finally:
        if temporary and str(path).startswith(str(TMP)):
            Path(path).unlink(missing_ok=True)

if HAS_AUDIO:
    from concurrent.futures import ThreadPoolExecutor
    aud_rows = db[db.camera == AUDIO_CAMERA].to_dict('records')
    with ThreadPoolExecutor(4) as ex:                      # ffmpeg runs as separate processes
        status = list(tqdm(ex.map(audio_one, aud_rows), total=len(aud_rows)))
    print(pd.Series(status).str.split(':').str[0].value_counts().to_dict())
else:
    print('Skipped: no audio in MCD videos.')

Skipped: no audio in MCD videos.


In [8]:
# Load the audio cache -> session table, per-patient features, per-second log-mel for section 5
AUD, AUD_COLS, AUD_COMPACT = {}, [], []
if HAS_AUDIO:
    srows = []
    for f in sorted(AUD_DIR.glob('*.npz')):
        pid, step = f.stem.rsplit('_', 1)
        with np.load(f) as z:
            srows.append(dict(patient_id=int(pid), step=step, **dict(zip(z['names'], z['vals']))))
            AUD[f'{pid}_{step}'] = z['mel']
    A = pd.DataFrame(srows)
    A.to_csv(FEAT_DIR / 'audio_session_features.csv', index=False)
    names = [c for c in A.columns if c not in ('patient_id', 'step')]

    # does the audio capture breathing? compare with the measured respiratory rate
    chk = A.merge(db[db.camera == AUDIO_CAMERA][['patient_id', 'step', 'respiratory']], on=['patient_id', 'step'])
    rho = spearmanr(chk.resp_rate, chk.respiratory, nan_policy='omit')[0]
    print(f'audio breathing rate vs measured respiratory rate: Spearman {rho:+.3f} '
          f'(median {chk.resp_rate.median():.1f} vs {chk.respiratory.median():.1f} /min)')

    # patient level: average of both sessions (trait), plus rest-session breathing
    Ap = A.groupby('patient_id')[names].mean().add_prefix('aud_')
    Ap['aud_resp_rate_rest'] = A[A.step == 'before'].set_index('patient_id').resp_rate
    P = P.drop(columns=[c for c in P.columns if c.startswith('aud_')]).join(Ap)
    AUD_COLS = list(Ap.columns)
    AUD_COMPACT = [c for c in ('aud_rms_mean', 'aud_active_frac', 'aud_resp_rate', 'aud_resp_peak',
                               'aud_centroid_mean', 'aud_flatness_mean') if c in P.columns]
    print('sessions with audio:', len(A), '| patients:', P[AUD_COLS[0]].notna().sum(), '| audio features:', len(AUD_COLS))
else:
    print('No audio features (HAS_AUDIO = False).')

No audio features (HAS_AUDIO = False).


## 3. Recording-context control (date and time of recording)

Audio, and also lighting, mostly describe the **recording environment**. If stress varied with *when* people were recorded (for example during exam periods), environment features could predict stress through the recording period rather than the person.

This section takes the recording date and time from the meta files and tests whether they predict stress. They're a control, not a usable predictor. If they matter, audio has to beat **demographics + context**, not just demographics.

In [9]:
TS_RE = re.compile(r'(\d{4}-\d{2}-\d{2}[ T]\d{2}:\d{2}:\d{2}(?:[.,]\d+)?)')
def first_timestamp(path):
    try:
        with open(path, errors='ignore') as fh:
            for ln in fh:
                m = TS_RE.search(ln)
                if m:
                    return pd.Timestamp(m.group(1).replace(',', '.'))
    except Exception:
        pass
    return pd.NaT

rest_rows = front[front.step == 'before']
ts = pd.Series({r.patient_id: first_timestamp(DATA_ROOT / r.meta) for r in rest_rows.itertuples()})
C = pd.DataFrame({'rec_day': (ts - ts.min()).dt.total_seconds() / 86400,
                  'rec_hour': ts.dt.hour + ts.dt.minute / 60,
                  'rec_weekday': ts.dt.weekday})
P = P.drop(columns=[c for c in C.columns if c in P.columns]).join(C)
CONTEXT = list(C.columns)
print('recordings dated:', ts.notna().sum(), '| from', ts.min(), 'to', ts.max())
pc_ctx = pd.DataFrame([partial_spearman(P, c) for c in CONTEXT])
print(pc_ctx.round(3).to_string(index=False))
monthly = P.groupby(ts.dt.to_period('M').reindex(P.index)).stress.agg(['count', 'mean']).round(2)
print('\nPSM-25 by recording month:\n', monthly.to_string())

recordings dated: 600 | from 2023-10-17 09:47:00.571132 to 2023-12-06 13:35:21.804858
    feature   n    rho     lo    hi     p
    rec_day 600  0.027 -0.054 0.107 0.515
   rec_hour 600 -0.012 -0.092 0.068 0.764
rec_weekday 600  0.066 -0.014 0.146 0.107

PSM-25 by recording month:
          count  mean
2023-10    246  2.97
2023-11    284  3.20
2023-12     70  2.64


## 4. rPPG + audio ablation (classical models)

Same patients, splits and models (Ridge, RandomForest, demographics-then-residual) for every set.
- rPPG = week-3 core features + week-4 within-person features.
- Audio = all features, plus a pre-specified compact set of 6 (loudness, activity, breathing rate and strength, spectral centroid, flatness) to limit overfitting.

**Read:** audio or rPPG helps only if the gain CI vs demographics excludes 0. It should also still hold with the recording-context control included.

In [10]:
SETS = {
    'demographics':                      DEMO,
    'demo + context (control)':          DEMO + CONTEXT,
    'rPPG only':                         RPPG_ALL,
    'demo + rPPG':                       DEMO + RPPG_ALL,
}
if HAS_AUDIO:
    SETS.update({
        'audio only':                        AUD_COLS,
        'demo + audio (compact)':            DEMO + AUD_COMPACT,
        'demo + audio (all)':                DEMO + AUD_COLS,
        'rPPG + audio (no demographics)':    RPPG_ALL + AUD_COMPACT,
        'demo + rPPG + audio':               DEMO + RPPG_ALL + AUD_COMPACT,
        'demo + context + rPPG + audio':     DEMO + CONTEXT + RPPG_ALL + AUD_COMPACT,
    })
abl = run_ladder(P, SETS)
print(abl.round(3).to_string(index=False))
abl.to_csv(RES_DIR / 'ablation_rppg_audio.csv', index=False)

  0%|          | 0/4 [00:00<?, ?it/s]

                     set    model  n_feat  test_mae  cv_mae  skill  skill_lo  skill_hi   gain  gain_lo  gain_hi
 constant (train median)        -       0     1.117   1.154    NaN       NaN       NaN    NaN      NaN      NaN
            demographics    Ridge       3     1.143   1.149  0.425    -1.821     2.527  0.000    0.000    0.000
            demographics       RF       3     1.170   1.161 -0.590    -2.680     1.425 -1.015   -2.156    0.109
demo + context (control)    Ridge       6     1.142   1.154  0.036    -1.985     1.990 -0.389   -0.989    0.260
demo + context (control)       RF       6     1.082   1.092  5.354     2.310     8.518  4.928    2.214    7.601
demo + context (control) Residual       6     1.143   1.151  0.286    -1.947     2.379 -0.139   -0.245   -0.034
               rPPG only    Ridge      13     1.129   1.166 -0.978    -1.309    -0.666 -1.404   -3.421    0.657
               rPPG only       RF      13     1.155   1.188 -2.889    -4.269    -1.417 -3.314   -5.403  

In [11]:
# High-stress (>= 4.0) AUC and partial correlations of the audio features
yb = (P.stress >= 4.0).astype(int).values
p_ref = oof_proba(P[DEMO], yb)
auc_sets = {k: v for k, v in SETS.items() if k in ('demo + context (control)', 'demo + rPPG', 'demo + audio (compact)',
                                                   'demo + rPPG + audio', 'demo + context + rPPG + audio')}
auc = pd.DataFrame([dict(set='demographics', **boot_auc(yb, p_ref, p_ref))] +
                   [dict(set=k, **boot_auc(yb, oof_proba(P[v], yb), p_ref)) for k, v in auc_sets.items()])
print('High stress (>= 4.0) AUC\n', auc.round(3).to_string(index=False))
auc.to_csv(RES_DIR / 'ablation_auc.csv', index=False)

if HAS_AUDIO:
    pca = pd.DataFrame([partial_spearman(P, c) for c in AUD_COLS])
    pca['q'] = bh(pca.p)
    print('\nAudio features vs stress (age, sex, BMI removed), strongest 10:')
    print(pca.reindex(pca.rho.abs().sort_values(ascending=False).index).head(10).round(3).to_string(index=False))
    pca.to_csv(RES_DIR / 'audio_partial_correlations.csv', index=False)

High stress (>= 4.0) AUC
                      set   auc  auc_lo  auc_hi  d_auc   d_lo  d_hi
            demographics 0.602   0.557   0.649  0.000  0.000 0.000
demo + context (control) 0.605   0.559   0.650  0.003 -0.031 0.035
             demo + rPPG 0.607   0.560   0.655  0.005 -0.031 0.039


## 5. Transformers: pulse, audio, and rPPG + audio fusion

One small transformer encoder (2 layers, 4 heads, d = 64) with a CLS token and a before/after session embedding. Inputs are 60 s windows cut into **1 s tokens**:
- **Pulse tokens:** 30 samples per second of the cached rPPG signal, z-scored per window (no skin-tone or brightness level, so no appearance shortcut).
- **Audio tokens:** the 40-band log-mel of each second, standardised.
- **Fusion:** pulse and audio tokens for the same 60 s, with a modality embedding. Attention can relate breathing sounds to the pulse.

Variants:
1. rPPG pulse.
2. Contact-PPG pulse (ceiling).
3. Audio only, if audio exists.
4. rPPG + audio fusion, if audio exists.

Training uses L1 loss (MAE) and early stopping on 10% of training patients. A patient's prediction is the mean over windows of both sessions. Evaluation uses the fixed split and 5-fold patient CV, with the constant and demographics refit in the same folds.

**A GPU is recommended** (Runtime → Change runtime type → T4). On CPU, set `RUN_CV = False`.

In [12]:
import torch, torch.nn as nn
torch.set_num_threads(N_WORKERS)
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FS = int(FS_RPPG)
WIN_SEC = 60                         # tokens per modality (1 s each)
EPOCHS, PATIENCE = 40, 8
WIN_PER_SESSION, BATCH = 6, 64
RUN_CV = True
RUN_PPG_CEILING = True
N_MEL = 40
if HAS_AUDIO and AUD:
    _all = np.concatenate([v.astype(np.float32) for v in AUD.values()])
    MEL_MU, MEL_SD = _all.mean(0), _all.std(0) + 1e-6
print('device:', DEV)

def session_items(kind, src, pid):
    # kind: 'pulse' | 'audio' | 'fusion'; returns [(pulse or None, mel or None, flag, n_seconds)]
    out = []
    for flag, step in ((0.0, 'before'), (1.0, 'after')):
        pul = SIG.get(f'{src}_{pid}_{step}') if kind in ('pulse', 'fusion') else None
        mel = AUD.get(f'{pid}_{step}') if kind in ('audio', 'fusion') else None
        if (kind in ('pulse', 'fusion') and pul is None) or (kind in ('audio', 'fusion') and mel is None):
            continue
        T = min(len(pul) // FS if pul is not None else 10 ** 9, len(mel) if mel is not None else 10 ** 9)
        if T >= WIN_SEC:
            out.append((pul, mel, flag, T))
    return out

def make_batch(items, starts):
    B = len(items)
    Pb = np.zeros((B, WIN_SEC, FS), np.float32)
    Ab = np.zeros((B, WIN_SEC, N_MEL), np.float32)
    Fb = np.zeros(B, np.float32)
    for i, ((pul, mel, flag, T), s0) in enumerate(zip(items, starts)):
        if pul is not None:
            w = pul[s0 * FS:(s0 + WIN_SEC) * FS].astype(np.float32)
            Pb[i] = ((w - w.mean()) / (w.std() + 1e-6)).reshape(WIN_SEC, FS)
        if mel is not None:
            Ab[i] = (mel[s0:s0 + WIN_SEC].astype(np.float32) - MEL_MU) / MEL_SD
        Fb[i] = flag
    return [torch.from_numpy(a).to(DEV) for a in (Pb, Ab, Fb)]

class StressTransformer(nn.Module):
    def __init__(self, use_pulse, use_audio, d=64, heads=4, layers=2, ff=128, drop=0.2):
        super().__init__()
        self.use_pulse, self.use_audio = use_pulse, use_audio
        if use_pulse:
            self.pulse_in = nn.Sequential(nn.Linear(FS, d), nn.LayerNorm(d))
        if use_audio:
            self.audio_in = nn.Sequential(nn.Linear(N_MEL, d), nn.LayerNorm(d))
        self.pos = nn.Parameter(torch.randn(1, WIN_SEC, d) * 0.02)
        self.mod = nn.Parameter(torch.randn(2, d) * 0.02)
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.sess = nn.Embedding(2, d)
        layer = nn.TransformerEncoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.head = nn.Sequential(nn.LayerNorm(d), nn.Dropout(drop), nn.Linear(d, 1))
    def forward(self, p, a, flag):
        toks = []
        if self.use_pulse:
            toks.append(self.pulse_in(p) + self.pos + self.mod[0])
        if self.use_audio:
            toks.append(self.audio_in(a) + self.pos + self.mod[1])
        x = torch.cat(toks, 1)
        cls = self.cls.expand(len(x), -1, -1) + self.sess(flag.long()).unsqueeze(1)
        return self.head(self.enc(torch.cat([cls, x], 1))[:, 0]).squeeze(-1)

@torch.no_grad()
def predict_patients(model, kind, src, pids, stride_sec=30):
    model.eval()
    out = {}
    for pid in pids:
        items = session_items(kind, src, pid)
        if not items:
            continue
        batch_items, starts = [], []
        for it in items:
            for s0 in range(0, it[3] - WIN_SEC + 1, stride_sec):
                batch_items.append(it); starts.append(s0)
        out[pid] = float(model(*make_batch(batch_items, starts)).mean().cpu())
    return out

def train_model(kind, src, train_pids, y_map, seed=0):
    rng = np.random.default_rng(seed); torch.manual_seed(seed)
    pids = np.array([p for p in train_pids if session_items(kind, src, p)])
    rng.shuffle(pids)
    n_val = max(5, int(0.1 * len(pids)))
    val, fit = pids[:n_val], pids[n_val:]
    yt = np.array([y_map[p] for p in fit])
    mu, sd = float(np.median(yt)), float(yt.std() + 1e-6)
    items = [(it, (y_map[p] - mu) / sd) for p in fit for it in session_items(kind, src, p)]
    model = StressTransformer(kind in ('pulse', 'fusion'), kind in ('audio', 'fusion')).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=1e-2)
    best, best_state, bad, ep = np.inf, None, 0, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        order = rng.permutation(len(items) * WIN_PER_SESSION) % len(items)
        for b in range(0, len(order), BATCH):
            chunk = [items[i] for i in order[b:b + BATCH]]
            if len(chunk) < 2:
                continue
            starts = [rng.integers(0, it[3] - WIN_SEC + 1) for it, _ in chunk]
            pb, ab, fb = make_batch([it for it, _ in chunk], starts)
            yb_t = torch.tensor([t for _, t in chunk], dtype=torch.float32, device=DEV)
            loss = nn.functional.l1_loss(model(pb, ab, fb), yb_t)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        pv = predict_patients(model, kind, src, val)
        mae = np.mean([abs(pv[p] * sd + mu - y_map[p]) for p in pv])
        if mae < best - 1e-4:
            best, bad = mae, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, mu, sd, ep

def model_predict(kind, src, train_pids, test_pids, y_map, seed=0):
    model, mu, sd, ep = train_model(kind, src, train_pids, y_map, seed)
    pv = predict_patients(model, kind, src, test_pids)
    return {p: v * sd + mu for p, v in pv.items()}, ep

device: cuda


In [14]:
VARIANTS = [('Transformer: rPPG pulse', 'pulse', 'rppg')]
if RUN_PPG_CEILING:
    VARIANTS.append(('Transformer: contact-PPG pulse (ceiling)', 'pulse', 'ppg'))
if HAS_AUDIO and AUD:
    VARIANTS += [('Transformer: audio', 'audio', None), ('Transformer: rPPG + audio fusion', 'fusion', 'rppg')]

y_map = P.stress.to_dict()
pids_all, ypat = P.index.values, P.stress.values
tr_ids, te_ids = train_test_split(pids_all, test_size=0.2, random_state=SPLIT_SEED)
tf_rows = []
for name, kind, src in VARIANTS:
    t0 = time.time()
    pred, ep = model_predict(kind, src, tr_ids, te_ids, y_map, seed=0)
    med = np.median([y_map[p] for p in tr_ids])
    yt = np.array([y_map[p] for p in te_ids]); pt = np.array([pred.get(p, med) for p in te_ids])
    tf_rows.append(dict(variant=name, split='fixed (rs=43)', mae=np.mean(np.abs(yt - pt)),
                        const_mae=np.mean(np.abs(yt - med)), spearman=spearmanr(pt, yt)[0],
                        pred_sd=pt.std(), epochs=ep, minutes=(time.time() - t0) / 60))
    print({k: (round(v, 3) if isinstance(v, float) else v) for k, v in tf_rows[-1].items()})

if RUN_CV:
    oof = {name: np.zeros(len(pids_all)) for name, _, _ in VARIANTS}
    e0f, edf = np.zeros(len(pids_all)), np.zeros(len(pids_all))
    for k, (a, b) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=100).split(pids_all)):
        med = np.median(ypat[a])
        e0f[b] = np.abs(ypat[b] - med)
        edf[b] = np.abs(ypat[b] - MODELS['Ridge']().fit(P.iloc[a][DEMO], ypat[a]).predict(P.iloc[b][DEMO]))
        for name, kind, src in VARIANTS:
            pred, _ = model_predict(kind, src, pids_all[a], pids_all[b], y_map, seed=k)
            oof[name][b] = [pred.get(p, med) for p in pids_all[b]]
        print(f'fold {k + 1}/{N_FOLDS} done')
    for name, _, _ in VARIANTS:
        e = np.abs(oof[name] - ypat)
        tf_rows.append(dict(variant=name, split=f'{N_FOLDS}-fold CV', mae=e.mean(), const_mae=e0f.mean(),
                            spearman=spearmanr(oof[name], ypat)[0], pred_sd=oof[name].std(),
                            **{k: 100 * v for k, v in boot_skill(e, e0f, edf).items()}))
tf_res = pd.DataFrame(tf_rows)
print(tf_res.round(3).to_string(index=False))
tf_res.to_csv(RES_DIR / 'transformer_results.csv', index=False)

{'variant': 'Transformer: rPPG pulse', 'split': 'fixed (rs=43)', 'mae': np.float64(1.144), 'const_mae': np.float64(1.117), 'spearman': np.float64(-0.239), 'pred_sd': np.float64(0.083), 'epochs': 13, 'minutes': 0.762}
{'variant': 'Transformer: contact-PPG pulse (ceiling)', 'split': 'fixed (rs=43)', 'mae': np.float64(1.123), 'const_mae': np.float64(1.117), 'spearman': np.float64(0.037), 'pred_sd': np.float64(0.031), 'epochs': 12, 'minutes': 0.373}
fold 1/5 done
fold 2/5 done
fold 3/5 done
fold 4/5 done
fold 5/5 done
                                 variant         split   mae  const_mae  spearman  pred_sd  epochs  minutes  skill  skill_lo  skill_hi   gain  gain_lo  gain_hi
                 Transformer: rPPG pulse fixed (rs=43) 1.144      1.117    -0.239    0.083    13.0    0.762    NaN       NaN       NaN    NaN      NaN      NaN
Transformer: contact-PPG pulse (ceiling) fixed (rs=43) 1.123      1.117     0.037    0.031    12.0    0.373    NaN       NaN       NaN    NaN      NaN      NaN


## 6. Comparison with the literature

Rows marked *reported* are copied from the papers; rows marked *this work* are filled in from the results above and in weeks 3–4.

**Note on comparability.** The studies with strong results all classify **acute, induced stress within the same people**: rest vs speech or arithmetic (UBFC-Phys), or per-task self-reports (StressID). MCD-rPPG asks something different and harder: regress a **trait questionnaire** (PSM-25) from a 3-minute resting recording. Split design also matters a lot. Several high numbers come from random or sample-level splits, and those are not comparable to patient-level evaluation.

In [15]:
lit = pd.DataFrame([
    dict(source='MCD-rPPG paper (arXiv 2508.17924)', type='reported', dataset='MCD-rPPG (600)',
         task='PSM-25 regression (trait)', input='facial ROI colour signals, 1-D FPN (multi-task)',
         protocol='split not stated', metric='MAE', result='1.07 (constant 1.20)'),
    dict(source='MCD-rPPG-baselines repo', type='reported', dataset='MCD-rPPG (600)',
         task='PSM-25 regression (trait)', input='video models', protocol='patient split, rs=43',
         metric='MAE', result='1.117-1.219 (constant 1.117)'),
    dict(source='UBFC-Phys (Meziati Sabour et al., IEEE TAC)', type='reported', dataset='UBFC-Phys (56)',
         task='rest vs stress task (T1 vs T2/T3)', input='remote PRV features, RBF SVM',
         protocol='stratified 7-fold CV', metric='accuracy', result='85.48% (contact PRV 75.00%)'),
    dict(source='Sensors 2024, 24(4):1096', type='reported', dataset='UBFC-Phys',
         task='T1 vs T2', input='rPPG HR (CHROM), 1-D CNN', protocol='random 80/10/10 (not subject-wise)',
         metric='accuracy', result='up to 95.83%'),
    dict(source='StressID (Chaptoukaev et al., NeurIPS 2023)', type='reported', dataset='StressID (65)',
         task='binary self-reported stress per task', input='physiological / video (AUs, gaze) / audio',
         protocol='subject-wise 90/10, 8 splits', metric='F1',
         result='physio 0.73, video 0.70, audio 0.70, video+audio 0.76, all 0.72'),
])

CONST_MAE = abl.iloc[0].test_mae if 'abl' in globals() else 1.117
ours = [('rPPG core features, Ridge (week 3)', 'MAE', '1.125'),
        ('rPPG within-person features (week 4)', 'MAE', '1.140'),
        ('1-D CNN on rPPG pulse (week 4)', 'MAE', '1.134')]
if 'abl' in globals():
    for s in ('demo + rPPG', 'demo + audio (compact)', 'demo + rPPG + audio'):
        r = abl[abl.set == s]
        if len(r):
            r = r.sort_values('cv_mae').iloc[0]
            ours.append((f'{s} ({r.model}, week 5)', 'MAE', f'{r.test_mae:.3f}; CV gain vs demo {r.gain:+.2f}%'))
if 'tf_res' in globals():
    for r in tf_res[tf_res.split.str.startswith('fixed')].itertuples():
        ours.append((f'{r.variant} (week 5)', 'MAE', f'{r.mae:.3f}'))
lit = pd.concat([lit, pd.DataFrame([dict(source=s, type='this work', dataset='MCD-rPPG (600)',
                                         task='PSM-25 regression (trait)', input=s.split(' (')[0],
                                         protocol='patient split rs=43 + 5-fold CV', metric=m,
                                         result=v + f' (constant {CONST_MAE:.3f})') for s, m, v in ours])], ignore_index=True)
pd.set_option('display.max_colwidth', 80)
print(lit[['source', 'type', 'task', 'protocol', 'metric', 'result']].to_string(index=False))
lit.to_csv(RES_DIR / 'literature_comparison.csv', index=False)

                                           source      type                                 task                           protocol   metric                                                          result
                MCD-rPPG paper (arXiv 2508.17924)  reported            PSM-25 regression (trait)                   split not stated      MAE                                            1.07 (constant 1.20)
                          MCD-rPPG-baselines repo  reported            PSM-25 regression (trait)               patient split, rs=43      MAE                                    1.117-1.219 (constant 1.117)
      UBFC-Phys (Meziati Sabour et al., IEEE TAC)  reported    rest vs stress task (T1 vs T2/T3)               stratified 7-fold CV accuracy                                     85.48% (contact PRV 75.00%)
                         Sensors 2024, 24(4):1096  reported                             T1 vs T2 random 80/10/10 (not subject-wise) accuracy                                        

## 7. Next dataset: readiness check

| Dataset | People | Stress label | Video | Audio | Physiological reference | Fit for |
|---|---|---|---|---|---|---|
| **StressID** | 65 (video 54, audio 55) | per-task self-report (binary / 3-class) | yes | **yes** (7 speaking tasks) | ECG, EDA, respiration | rPPG + audio, transformer fusion |
| **UBFC-Phys** | 56 | rest vs speech vs arithmetic (+ anxiety scores) | yes, 35 fps | not described | BVP 64 Hz, EDA 4 Hz | rPPG stress, within-person |

Both are within-person, induced-stress designs, where rPPG is expected to work. StressID is the natural place to test **rPPG + audio**, because participants speak.

The pipeline carries over: ROI trace extraction (week 3), POS signal cache (week 4), within-person features (week 4), and the section 5 transformer (for StressID, swap PSM-25 for task labels and use leave-subjects-out splits).

The cell below only checks whether either dataset is already on Drive.

In [16]:
for name in ('StressID', 'UBFC-Phys', 'UBFC_Phys'):
    d = ROOT.parent / name
    if d.exists():
        files = list(d.rglob('*'))
        print(f'{name}: found at {d} | {len(files)} files | e.g. {[f.name for f in files[:5]]}')
    else:
        print(f'{name}: not on Drive yet ({d}) -> request access, then upload/extract there')

StressID: not on Drive yet (/content/drive/MyDrive/StressID) -> request access, then upload/extract there
UBFC-Phys: not on Drive yet (/content/drive/MyDrive/UBFC-Phys) -> request access, then upload/extract there
UBFC_Phys: not on Drive yet (/content/drive/MyDrive/UBFC_Phys) -> request access, then upload/extract there


## 8. Summary for the weekly report

In [17]:
print(f"Patients: {len(P)} | audio in MCD videos: {HAS_AUDIO}" + (f" ({AUDIO_CAMERA})" if HAS_AUDIO else ''))
print('\nRecording context vs stress (partial Spearman):',
      ', '.join(f'{r.feature} {r.rho:+.3f} [{r.lo:+.3f}, {r.hi:+.3f}]' for r in pc_ctx.itertuples()))
const = abl.iloc[0]
print(f"\nConstant: fixed-split MAE {const.test_mae:.3f} | CV MAE {const.cv_mae:.3f}")
for s in SETS:
    r = abl[abl.set == s]
    if len(r):
        r = r.sort_values('cv_mae').iloc[0]
        print(f"{s:34s} ({r.model:8s}) MAE {r.test_mae:.3f} | CV skill {r.skill:+.2f}% [{r.skill_lo:+.2f}, {r.skill_hi:+.2f}]"
              f" | gain vs demo {r.gain:+.2f}% [{r.gain_lo:+.2f}, {r.gain_hi:+.2f}]")
if 'tf_res' in globals():
    print()
    for r in tf_res.itertuples():
        extra = f" | skill {r.skill:+.2f}% [{r.skill_lo:+.2f}, {r.skill_hi:+.2f}]" if 'CV' in r.split else ''
        print(f"{r.variant:42s} {r.split:14s} MAE {r.mae:.3f} vs constant {r.const_mae:.3f} | "
              f"Spearman {r.spearman:+.3f} | prediction SD {r.pred_sd:.3f}{extra}")
print('\nRead: helps only if the gain (or skill) CI excludes 0. Prediction SD << 1.46 (label SD) means the model'
      ' collapsed to the median, i.e. found nothing to learn.')

Patients: 600 | audio in MCD videos: False

Recording context vs stress (partial Spearman): rec_day +0.027 [-0.054, +0.107], rec_hour -0.012 [-0.092, +0.068], rec_weekday +0.066 [-0.014, +0.146]

Constant: fixed-split MAE 1.117 | CV MAE 1.154
demographics                       (Ridge   ) MAE 1.143 | CV skill +0.43% [-1.82, +2.53] | gain vs demo +0.00% [+0.00, +0.00]
demo + context (control)           (RF      ) MAE 1.082 | CV skill +5.35% [+2.31, +8.52] | gain vs demo +4.93% [+2.21, +7.60]
rPPG only                          (Ridge   ) MAE 1.129 | CV skill -0.98% [-1.31, -0.67] | gain vs demo -1.40% [-3.42, +0.66]
demo + rPPG                        (Residual) MAE 1.143 | CV skill +0.34% [-1.91, +2.43] | gain vs demo -0.09% [-0.18, +0.00]

Transformer: rPPG pulse                    fixed (rs=43)  MAE 1.144 vs constant 1.117 | Spearman -0.239 | prediction SD 0.083
Transformer: contact-PPG pulse (ceiling)   fixed (rs=43)  MAE 1.123 vs constant 1.117 | Spearman +0.037 | prediction SD 0.031
